# Chapter 03 — Models and Thinking Level

**Companion to *Pi Agents*** · [`Pi Agents` chapter 03](https://programmer.ie/books/pi/03-chapter/)

| | |
|---|---|
| Chapter | `03-chapter.md` · weight 3 · `/books/pi/03-chapter/` |
| Notebook | `notebooks/pi/03-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Four credential sources are configured at once. **Which one does Pi
actually use** — and does *how* you wrote the key change anything the agent does?

The second question is the chapter's real one. It is easy to believe that a key
fetched by a shell command is somehow a different agent from a key typed into a
file. The chapter's evidence says otherwise, and that is checkable.


## What this notebook establishes

- Credential resolution is **first-source-wins** in a fixed order: a runtime key, then `auth.json`, then an `apiKey` in `models.json`, then the environment variable. The sources after the winner are never consulted.
- A model listed in `models.json` **parses and loads** while remaining **unavailable** until its provider has some credential — and a dummy key is enough to change that.
- Three ways of *writing* a provider key — a literal, `$NAME` resolved from the environment, and `!command` executed by a shell — produce three different credential strings and an **identical agent transcript**: same requests, same blocked call, same roles.

## What this notebook does **not** establish

- **No credential is real.** `ENV-KEY`, `STORED`, `MODELS`, `RUNTIME` and the three faux-provider keys are literals written by this notebook. No network request is made and nothing is billed.
- **Nothing about model quality, pricing or the model catalogue.** Those are version-sensitive and are not exercised here.
- **Nothing about `samplingParams` per thinking level, virtual models or cache warming.** The chapter names them; none is run in this notebook.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — Pi's real `ModelRuntime` against real files, then a real `AgentSession` with the chapter's guard extension.
* **Evidence scope:** `in_process_runtime`. The provider is the scripted faux model; the credential machinery, the model runtime and the tool gate are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(3))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/03-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Experiment 1: which source wins

Four sources, removed one at a time. The winner's *identity* is the literal string
that source contributed, which makes the result impossible to confuse with a
default.

This is Pi's `ModelRuntime` reading real `auth.json` and `models.json` files from
a fresh temporary directory — the same call path the chapter's test uses.

In [3]:
RESOLVE_SRC = r"""
import { mkdtempSync, writeFileSync } from "node:fs";
import { tmpdir } from "node:os";
import { join } from "node:path";
import { ModelRuntime } from "@earendil-works/pi-coding-agent";

const ENV = { ANTHROPIC_API_KEY: "ENV-KEY" };
export async function resolved(src: any) {
	const dir = mkdtempSync(join(tmpdir(), "pinb-ch03-"));
	if (src.auth) writeFileSync(join(dir, "auth.json"), JSON.stringify({ anthropic: { type: "api_key", key: src.auth } }));
	if (src.models) writeFileSync(join(dir, "models.json"), JSON.stringify({ providers: { anthropic: { apiKey: src.models } } }));
	const rt = await ModelRuntime.create({
		authPath: join(dir, "auth.json"),
		modelsPath: src.models ? join(dir, 'models.json') : null,
		refreshOnCreate: false,
	});
	if (src.runtime) await rt.setRuntimeApiKey('anthropic', src.runtime);
	const r: any = await rt.getAuth('anthropic', { env: src.env ? ENV : {} } as any);
	return (r?.auth?.apiKey ?? null) as string | null;
}
"""

DRIVER = RESOLVE_SRC + r"""
const all = { runtime: 'RUNTIME', auth: 'STORED', models: 'MODELS', env: true };
const ladder: [string, string | null][] = [
	['all four sources', await resolved(all)],
	['drop the runtime key', await resolved({ ...all, runtime: undefined })],
	['drop auth.json too', await resolved({ ...all, runtime: undefined, auth: undefined })],
	['drop models.json too', await resolved({ ...all, runtime: undefined, auth: undefined, models: undefined })],
	['drop the environment too', await resolved({})],
];

// A model that parses but cannot be selected until its provider authenticates.
const dir = mkdtempSync(join(tmpdir(), "pinb-ch03-avail-"));
const write = (apiKey?: string) =>
	writeFileSync(
		join(dir, "models.json"),
		JSON.stringify({ providers: { mylocal: { baseUrl: "http://localhost:1/v1", api: "openai-completions", ...(apiKey ? { apiKey } : {}), models: [{ id: "my-model" }] } } }),
	);
const probe = async () => {
	const rt = await ModelRuntime.create({ authPath: join(dir, 'auth.json'), modelsPath: join(dir, 'models.json'), refreshOnCreate: false });
	return { modelLoads: Boolean(rt.getModel('mylocal', 'my-model')), available: (await rt.getAvailable('mylocal')).map((m: any) => m.id) };
};
write();
const withoutKey = await probe();
write('ollama');
const withDummyKey = await probe();

console.log(JSON.stringify({ ladder, withoutKey, withDummyKey }));
"""

creds = pinb.driver(DRIVER, label="ch03-credentials", timeout=180)
print(pinb.table(["sources configured", "resolved apiKey"],
                 [[label, value if value is not None else "(none)"] for label, value in creds["ladder"]], indent=""))

sources configured  resolved apiKey
------------------------  ---------------
all four sources  RUNTIME
drop the runtime key  STORED
drop auth.json too  MODELS
drop models.json too  ENV-KEY
drop the environment too  (none)


Read the ladder top to bottom: each row removes one source and the winner
changes. That is **first-source-wins**, and it is not a merge or a fallback with
preferences — once a source yields a credential, nothing below it is consulted.

The second experiment is the quieter one. A model in `models.json` with no
`apiKey` **loads** — `getModel` returns it — and is still absent from the
available list. Adding the string `ollama` changes nothing about the model and
everything about whether you can select it.

In [4]:
print("models.json entry, before and after a credential is added:")
print(pinb.table(["models.json", "getModel() returns it", "getAvailable() lists it"],
                 [["no apiKey", creds["withoutKey"]["modelLoads"], creds["withoutKey"]["available"] or "(empty)"],
                  ["apiKey: ollama", creds["withDummyKey"]["modelLoads"], creds["withDummyKey"]["available"]]], indent=""))

winners = [v for _, v in creds["ladder"]]
print()
pinb.show_checks([
    pinb.check("runtime wins when all four are present", winners[0] == "RUNTIME", str(winners[0])),
    pinb.check("auth.json wins once the runtime key is gone", winners[1] == "STORED", str(winners[1])),
    pinb.check("models.json wins next", winners[2] == "MODELS", str(winners[2])),
    pinb.check("the environment is the last resort", winners[3] == "ENV-KEY", str(winners[3])),
    pinb.check("with nothing configured there is no credential", winners[4] is None, str(winners[4])),
    pinb.check("a model with no provider credential loads but is not available",
               creds["withoutKey"]["modelLoads"] and not creds["withoutKey"]["available"],
               f"loads={creds['withoutKey']['modelLoads']}, available={creds['withoutKey']['available']}"),
    pinb.check("adding a dummy apiKey makes it available",
               "my-model" in creds["withDummyKey"]["available"],
               str(creds["withDummyKey"]["available"])),
])

models.json entry, before and after a credential is added:
models.json  getModel() returns it  getAvailable() lists it
--------------  ---------------------  -----------------------
no apiKey  True  (empty)
apiKey: ollama  True  ['my-model']

== Assertions ==
  assertion  observed
----  --------------------------------------------------------------  ------------------------
PASS  runtime wins when all four are present  RUNTIME
PASS  auth.json wins once the runtime key is gone  STORED
PASS  models.json wins next  MODELS
PASS  the environment is the last resort  ENV-KEY
PASS  with nothing configured there is no credential  None
PASS  a model with no provider credential loads but is not available  loads=True, available=[]
PASS  adding a dummy apiKey makes it available  ['my-model']

7/7 assertions held.


## Experiment 2: provider configuration is not agent behaviour

This is the chapter's sharper claim, and it needs a real agent to test. The same
extension — the chapter 16 guard that refuses a force push — is loaded into three
real `AgentSession`s that differ **only** in how the faux provider's key is
spelled.

Three credential strings. One transcript.

In [5]:
DRIVER2 = r"""
import { fauxAssistantMessage, fauxText, fauxToolCall } from "@earendil-works/pi-ai";
import { makeSession } from "../harness/session.ts";
import guard from "../ch16-events/guard.ts";

const bash = (command: string) => fauxAssistantMessage([fauxToolCall('bash', { command })], { stopReason: 'toolUse' });

async function transcriptShape(apiKey: string, env: Record<string, string>) {
	Object.assign(process.env, env);
	const h = await makeSession({ extensions: [guard], apiKey });
	h.faux.setResponses([bash('git push --force origin main'), bash('echo allowed'), fauxAssistantMessage([fauxText('done')])]);
	await h.session.prompt('go');
	const key = ((await h.modelRuntime.getAuth('faux')) as any)?.auth?.apiKey as string;
	const results = h.session.messages.filter((m) => m.role === 'toolResult')
		.map((m: any) => ({ isError: m.isError, text: m.content[0].text.slice(0, 60) }));
	const roles = h.session.messages.map((m) => m.role);
	const requests = h.faux.state.callCount;
	h.dispose();
	return { key, shape: { results, roles, requests } };
}

const literal = await transcriptShape('a-literal-key', {});
const fromEnv = await transcriptShape('$PINB_TEST_KEY', { PINB_TEST_KEY: 'from-the-environment' });
const fromCommand = await transcriptShape('!echo from-a-command', {});

console.log(JSON.stringify({ literal, fromEnv, fromCommand }));
"""
shapes = pinb.driver(DRIVER2, label="ch03-key-spelling", timeout=240)
print(pinb.table(["key written as", "resolved credential", "requests", "tool results (in order)"],
                 [[label, shapes[k]["key"].strip(), shapes[k]["shape"]["requests"],
                   " | ".join(("ERR " if r["isError"] else "ok  ") + r["text"].splitlines()[0][:38] for r in shapes[k]["shape"]["results"])]
                  for label, k in [("literal", "literal"), ("$NAME", "fromEnv"), ("!command", "fromCommand")]], indent=""))

key written as  resolved credential  requests  tool results (in order)
--------------  --------------------  --------  --------------------------------------------------------
literal  a-literal-key  3  ERR Guard blocked: \bgit\s+push\b[^\n]*(-- | ok  allowed
$NAME  from-the-environment  3  ERR Guard blocked: \bgit\s+push\b[^\n]*(-- | ok  allowed
!command  from-a-command  3  ERR Guard blocked: \bgit\s+push\b[^\n]*(-- | ok  allowed


In [6]:
credentials = {"literal": shapes["literal"]["key"], "$NAME": shapes["fromEnv"]["key"], "!command": shapes["fromCommand"]["key"].strip()}

pinb.show_checks([
    pinb.check("three spellings produced three different credentials",
               len(set(credentials.values())) == 3, str(credentials)),
    pinb.check("the force push was blocked under every spelling",
               all(s["shape"]["results"][0]["isError"] for s in shapes.values()), "blocked in all three"),
    pinb.check("the ordinary command was allowed under every spelling",
               not any(s["shape"]["results"][1]["isError"] for s in shapes.values()), "allowed in all three"),
    pinb.check("the three transcripts are identical",
               shapes["literal"]["shape"] == shapes["fromEnv"]["shape"] == shapes["fromCommand"]["shape"],
               "roles, results and request count all equal"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------------------  --------------------------------------------------------------------------
PASS  three spellings produced three different credentials  {'literal': 'a-literal-key', '$NAME': 'from-the-environment', '!command':…
PASS  the force push was blocked under every spelling  blocked in all three
PASS  the ordinary command was allowed under every spelling  allowed in all three
PASS  the three transcripts are identical  roles, results and request count all equal

4/4 assertions held.


## Contrasting case: the credential is not the policy

The blocked force push is the visible part. What is interesting is that the
*credential string itself* never appears anywhere in the transcript. A key read
from a shell command, a key from an environment variable and a typed literal are
indistinguishable from the agent's point of view — which is the point. The guard
decided on the command, not on where the key came from.

### The chapter's own tests

Everything above is a focused driver. The chapter's seven tests assert the same
boundaries plus the availability result, and they run in about a second.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = 'ch03-credentials/credentials.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch03-credentials/credentials.test.ts', show="credential")

$ node --test ch03-credentials/credentials.test.ts
  PASS  7 passed, 0 failed, 0 skipped  in 1547 ms

  tests matching 'credential':
    ok   a stored auth.json credential beats the environment variable
    ok   the first source that yields a credential wins, and the rest are never consulted: runtime, then auth.json, then models.json, then environment


## Your turn: predict, then run

**Predict first.** `LADDER` keeps the runtime key present in the first three rows.
What should the resolved key be in each row, and does anything else change?

**Then change one input.** Add `models` to the last row, or remove the runtime key
from the first, and re-run.

**Predict the boundary.** Which row is the first at which the resolved value is
`None`?

In [8]:
LADDER = [
    ("runtime + auth", {"runtime": "RUNTIME", "auth": "STORED", "env": True}),
    ("runtime only", {"runtime": "RUNTIME"}),
    ("runtime + models", {"runtime": "RUNTIME", "models": "MODELS", "env": True}),
    ("nothing at all", {}),
]
print("predicted winners:", [src or "(none)" for _, src in LADDER])

import json as _json

EXERCISE = RESOLVE_SRC + r"""
const LADDER: [string, any][] = JSON.parse(process.env.NB_LADDER ?? "[]");
const out: [string, string | null][] = [];
for (const [label, src] of LADDER) out.push([label, await resolved(src)]);
console.log(JSON.stringify(out));
"""

out = pinb.driver(
    EXERCISE,
    label="ch03-ladder",
    timeout=180,
    env={"NB_LADDER": _json.dumps([[label, src] for label, src in LADDER])},
)
print()
for label, value in out:
    print(f"  {label:22s} -> {value if value is not None else '(none)'}")

assert all(v == "RUNTIME" for _, v in out[:3]), "a runtime key outranks every file and the environment"
assert out[3][1] is None
print("\nheld: while a runtime key exists, nothing else is consulted at all")

predicted winners: [{'runtime': 'RUNTIME', 'auth': 'STORED', 'env': True}, {'runtime': 'RUNTIME'}, {'runtime': 'RUNTIME', 'models': 'MODELS', 'env': True}, '(none)']



  runtime + auth         -> RUNTIME
  runtime only           -> RUNTIME
  runtime + models       -> RUNTIME
  nothing at all         -> (none)

held: while a runtime key exists, nothing else is consulted at all


## Interpretation: two different questions

The chapter's material separates cleanly, and conflating them is the mistake it
is written against.

| Question | Where it is answered | What decides it |
|---|---|---|
| *Which credential does Pi use?* | experiment 1 | a fixed four-step order, first match wins |
| *What does the agent do?* | experiment 2 | the transcript: policy, tools and the model |

Nothing in the second table mentions credentials. That is the finding: how you
supply a key is an operational detail, and the moment you start reasoning about
what an agent will *do*, you have left that question behind.

The practical decisions this supports:

1. **Debug the key, not the agent.** A provider that will not authenticate is a
   configuration question; the resolution order tells you which file to look at.
2. **Beware the "loads but is unavailable" state.** A model parsing out of
   `models.json` is not the same as a model you can select. The dummy-key row is
   the cheapest way to see the difference.
3. **Version-sensitive parts are named, not exercised.** `samplingParams` per
   thinking level arrived in 1.0.2 and applies to three APIs; the classifier list
   changed in 1.0.1; the `azure-openai-responses` *provider* was renamed to
   `azure` in 1.0.4 while the *API* id stayed. None of that is run here.

## Sources, execution mode and limitations

**Execution mode:** **Run** — Pi's `ModelRuntime` against real files, and a real `AgentSession` with the chapter 16 guard, plus the chapter's own test file.

**Evidence scope:** `in_process_runtime`. Credential resolution, the model runtime and the tool gate are Pi's; the model is the scripted faux provider.

### What was read or run

- **Ran** `ModelRuntime.create` + `getAuth` over fresh temporary `auth.json` / `models.json` files, varying which sources exist.
- **Ran** three real `AgentSession`s through `makeSession`, loading `ch16-events/guard.ts` unmodified.
- **Ran** `ch03-credentials/credentials.test.ts`.
- **Read** `examples/evidence.json`, chapter 3 rows (7 claims).

### Limitations

- **No credential here is real.** Each is a literal string written by this notebook; `!echo from-a-command` runs a shell echo. Nothing reaches a provider and nothing is billed.
- **The model is scripted**, so the identical-transcript result is a claim about the code around a model, not about model behaviour under three keys. This book does not claim a real model would behave identically, only that nothing in Pi's design gives it a reason to differ.
- **Model ids, the catalogue and pricing are version-sensitive** and are not exercised. The `ollama` row is a literal string used as a credential, not a connection to an Ollama instance.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
